# Global Gridded Climate Dataset: Overview & Trend Analysis
**1° × 1° monthly grid · 2000 – 2025 · 20 variables · 3 NASA GES DISC products**

| Product | Variables | Domain |
|---|---|---|
| **GLDAS Noah 0.25° (v2.1)** | 12 land-surface variables (evaporation, precipitation, air/skin temperature, soil moisture ×3, snow ×2, potential evaporation, shortwave radiation, wind) | land only, 60°S – 90°N |
| **GPM IMERG (v07)** | satellite precipitation | global, land + ocean |
| **MERRA-2 (M2TMNXSLV 5.12.4)** | 2 m air temperature, surface temperature, 2 m humidity, column water vapour, 10 m wind (u, v), surface pressure | global, land + ocean |

Data are built by `scripts/ingest/nasa_grid.py` (`run`, then `assemble`) into `data/nasa_grid/processed/`. Figures are saved to `docs/figures/nasa_grid/`.

**Contents**
1. Loading and integrity checks
2. Coverage: where do we have data?
3. Value distributions
4. Global time series and trends
5. Climate zones: trends by latitude band
6. Zonal structure: latitude × time
7. Global trend maps (every grid cell)
8. Cross-validation between products
9. Data-quality audit
10. Summary

**Trend method (identical everywhere).** monthly series → anomalies (subtract each calendar month's 2000–2025 mean) → annual means → OLS on 26 years (slope per decade, 95 % CI, p-value), cross-checked with Mann-Kendall and Theil-Sen. Because many series are tested at once, p-values are adjusted with Benjamini-Hochberg (FDR).

In [ ]:
import sys, warnings, time, json, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from cycler import cycler
from scipy import stats
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')

# ---------- visual theme ----------
PAL = dict(navy='#0b3954', red='#e4572e', teal='#2a9d8f', gold='#e9c46a', blue='#3a86ff', grey='#8d99ae', plum='#7b2cbf')
PKG_COLOR = {'gldas':PAL['teal'], 'imerg':PAL['blue'], 'merra2':PAL['plum']}
plt.rcParams.update({
    'figure.dpi':110, 'figure.facecolor':'white', 'axes.facecolor':'#fbfbfd',
    'axes.grid':True, 'grid.color':'#e3e6ee', 'grid.linewidth':.7,
    'axes.edgecolor':'#9aa5b1', 'axes.spines.top':False, 'axes.spines.right':False,
    'axes.titleweight':'bold', 'axes.titlesize':9.5, 'axes.titlecolor':PAL['navy'],
    'axes.labelcolor':'#333', 'font.size':9, 'legend.frameon':False,
    'axes.prop_cycle':cycler(color=[PAL['navy'],PAL['red'],PAL['teal'],PAL['gold'],PAL['blue'],PAL['plum']])})
pd.set_option('display.width',200); pd.set_option('display.max_columns',40)

def title(fig, text, y=1.01): fig.suptitle(text, fontsize=14, fontweight='bold', color=PAL['navy'], y=y)
def nice(df, gradient=None, cmap='Blues'):
    try:
        st = df.style.format(precision=3, thousands=',', na_rep='–')
        if gradient: st = st.background_gradient(cmap=cmap, subset=gradient)
        return st
    except Exception:
        return df

# ---------- paths and config ----------
ROOT = Path.cwd()
while not (ROOT/'scripts'/'ingest').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'scripts'/'ingest'))
import nasa_grid as ng                                   # variable catalog and data paths
GRID  = ng.PROC                                          # data/nasa_grid/processed/<variable>.csv
CACHE = ng.DATA/'analysis'/'cache'; CACHE.mkdir(parents=True, exist_ok=True)
FIG   = ROOT/'docs'/'figures'/'nasa_grid'; FIG.mkdir(parents=True, exist_ok=True)
def save_fig(fig, name): fig.savefig(FIG/f'{name}.png', dpi=110, bbox_inches='tight')
def cached(name, fn):
    p = CACHE/f'{name}.pkl'
    if p.exists(): return pickle.loads(p.read_bytes())
    r = fn(); p.write_bytes(pickle.dumps(r)); return r

VARS = ng.VARIABLES
KEYS = list(VARS)
pkg = lambda k: k.split('_')[0]

MONTHS = pd.period_range('2000-01','2025-12',freq='M'); T = len(MONTHS); NY = T//12
YEARS = np.arange(2000,2026)
LAT = np.arange(-89.5,90,1.0); LON = np.arange(-179.5,180,1.0)
LATM = LAT[:,None]*np.ones((1,360))
W = np.cos(np.deg2rad(LAT))[:,None]*np.ones((1,360))     # cell weight ~ area
date_to_idx = {f'{p.year}-{p.month:02d}-01':i for i,p in enumerate(MONTHS)}
U = {k: VARS[k]['units'] for k in KEYS}
print('Project root:', ROOT); print('Variables:', len(KEYS), '| months on time axis:', T)
pd.DataFrame({k:{'product':VARS[k]['short_name'],'variable':VARS[k]['var'],'units':VARS[k]['units']} for k in KEYS}).T

## 1. Loading: long CSV → data cube (time × lat × lon)

Each CSV has the columns `date, lat, lon, value, valid_frac`, and cells without data are simply absent. Every file is read in chunks and placed into an array of shape `(312, 180, 360)` where missing cells become `NaN`. Cubes are cached in `data/analysis/cache/`, so reruns are fast.

Integrity is checked while loading: row count, duplicates (`rows − filled cells`), bad dates.

In [ ]:
def load_cube(key, force=False):
    f, ff, st = CACHE/f'{key}.npy', CACHE/f'{key}_frac.npy', CACHE/f'{key}_stats.json'
    if f.exists() and ff.exists() and st.exists() and not force:
        return np.load(f, mmap_mode='r'), np.load(ff), json.loads(st.read_text())
    cube = np.full((T,180,360), np.nan, np.float32)
    fsum = np.zeros(64800); fcnt = np.zeros(64800)
    nrows = 0; nbad = 0; fmin = 1.0
    dt = {'lat':'float32','lon':'float32','value':'float32','valid_frac':'float32'}
    for ch in pd.read_csv(GRID/f'{key}.csv', usecols=['date','lat','lon','value','valid_frac'],
                          dtype=dt, chunksize=2_000_000):
        t = ch['date'].map(date_to_idx); ok = t.notna().values
        nbad += int((~ok).sum()); ch = ch[ok]; t = t[ok].astype(int).values
        i = np.rint(ch['lat'].values+89.5).astype(int); j = np.rint(ch['lon'].values+179.5).astype(int)
        cube[t,i,j] = ch['value'].values
        flat = i*360+j
        fsum += np.bincount(flat, weights=ch['valid_frac'].values, minlength=64800)
        fcnt += np.bincount(flat, minlength=64800)
        nrows += len(ch); fmin = min(fmin, float(ch['valid_frac'].min()))
    frac = (fsum/np.where(fcnt>0,fcnt,np.nan)).reshape(180,360).astype(np.float32)
    fin = np.isfinite(cube); vals = cube[fin]
    mp = int(fin.any(axis=(1,2)).sum()); p1,p99 = np.percentile(vals,[1,99])
    s = dict(rows=nrows, finite=int(fin.sum()), dup=nrows-int(fin.sum()), bad_dates=nbad,
             months=mp, cells_any=int(fin.any(0).sum()), cells_full=int((fin.sum(0)==mp).sum()),
             min=float(vals.min()), p1=float(p1), mean=float(vals.mean(dtype=np.float64)),
             p99=float(p99), max=float(vals.max()), std=float(vals.std(dtype=np.float64)),
             frac_min=fmin, size_mb=round((GRID/f'{key}.csv').stat().st_size/1e6,1))
    np.save(f, cube); np.save(ff, frac); st.write_text(json.dumps(s))
    return np.load(f, mmap_mode='r'), frac, s

In [ ]:
STATS, FRAC = {}, {}
t0 = time.time()
for k in KEYS:
    c, fr, s = load_cube(k); STATS[k] = s; FRAC[k] = fr; del c
    print(f'{k:24s} {s["rows"]:>11,} rows   {time.time()-t0:6.0f} s')
tab = pd.DataFrame(STATS).T
tab.insert(0,'units',[U[k] for k in tab.index])
cols = ['units','size_mb','rows','dup','bad_dates','months','cells_any','cells_full','frac_min','min','p1','mean','p99','max','std']
display(nice(tab[cols].astype({'rows':int,'dup':int,'bad_dates':int,'months':int,'cells_any':int,'cells_full':int}), gradient=['size_mb']))

**How to read the table.** `months` = months (out of 312) that contain at least one cell · `cells_any` = cells with at least one value · `cells_full` = cells with a value in **every** available month (the rest have temporal gaps) · `dup` = duplicate rows (must be 0) · `frac_min` = smallest valid area fraction of a cell (the pipeline threshold is 0.5).

In [ ]:
problems = []
for k,s in STATS.items():
    if s['dup']:       problems.append(f'{k}: {s["dup"]} duplicate rows')
    if s['bad_dates']: problems.append(f'{k}: {s["bad_dates"]} rows with a date outside 2000–2025')
    if s['cells_full'] != s['cells_any']: problems.append(f'{k}: {s["cells_any"]-s["cells_full"]} cells with temporal gaps')
    if s['months'] != 312: problems.append(f'{k}: {s["months"]} of 312 months present')
    if s['frac_min'] < 0.5-1e-6: problems.append(f'{k}: valid_frac below threshold ({s["frac_min"]:.3f})')
print('\n'.join(problems) if problems else '✔ No integrity problems found')

## 2. Coverage: where do we have data?

GLDAS covers **land only**; IMERG and MERRA-2 cover the whole globe including oceans. From here on `LAND` is the GLDAS land mask (cells where evaporation exists). It lets us compare products on identical cells.

In [ ]:
def show_map(ax, arr, title_, cmap='viridis', vmin=None, vmax=None, label='', norm=None, ticks=None):
    im = ax.imshow(arr, origin='lower', extent=[-180,180,-90,90], cmap=cmap, norm=norm,
                   vmin=None if norm is not None else vmin, vmax=None if norm is not None else vmax,
                   aspect='auto', interpolation='nearest')
    ax.set_title(title_); ax.grid(False); ax.set_xticks(range(-180,181,60)); ax.set_yticks(range(-90,91,30))
    cb = plt.colorbar(im, ax=ax, shrink=.85, pad=.02, ticks=ticks); cb.set_label(label); cb.outline.set_visible(False); return im

from matplotlib.colors import BoundaryNorm
COV_BOUNDS = [0,1,100,200,250,300,308,309,310,311,312,313]
COV_NORM = BoundaryNorm(COV_BOUNDS, ncolors=256)
COV_TICKS = [0,100,200,250,300,309,312]
LAND = np.isfinite(np.asarray(load_cube('gldas_evap')[0])).any(0)
fig, axs = plt.subplots(1,3, figsize=(18,4.4))
for ax,k in zip(axs, ['gldas_evap','imerg_precip','merra2_t2m']):
    n = np.isfinite(np.asarray(load_cube(k)[0])).sum(0).astype(float); n[n==0] = np.nan
    show_map(ax, n, f'{k}: months with data (max {int(np.nanmax(n))} of 312)', cmap='viridis',
             label='months (fixed scale 0–312)', norm=COV_NORM, ticks=COV_TICKS)
title(fig, 'Spatial coverage by product'); plt.tight_layout(); save_fig(fig, 'coverage_map'); plt.show()
la = np.where(LAND.any(1))[0]
print(f'GLDAS land mask: {LAND.sum()} cells, latitudes {LAT[la.min()]}° … {LAT[la.max()]}°')
print(f'Share of Earth surface (by area) covered by GLDAS: {(W*LAND).sum()/W.sum()*100:.1f} %')

In [ ]:
# GLDAS variables do not all share the same cell set (15,180 vs 15,332). Where is the difference?
sets = {k: np.isfinite(np.asarray(load_cube(k)[0])).any(0) for k in KEYS if k.startswith('gldas')}
big = sets['gldas_tair']; small = sets['gldas_evap']; diff = big & ~small
print('Cells only in the wider set (tair, precip, swdown, wind):', diff.sum(), '| reverse:', (small & ~big).sum())
ii,jj = np.where(diff)
print('Examples (lat, lon):', [(float(LAT[a]), float(LON[b])) for a,b in list(zip(ii,jj))[:12]])
if len(ii): print('Latitude range of the difference:', LAT[ii].min(), 'to', LAT[ii].max())
fig, ax = plt.subplots(figsize=(10,3.8)); show_map(ax, diff.astype(float), 'Cells present in tair/precip/swdown/wind but absent in evap/soil/swe …', cmap='Reds'); save_fig(fig, 'gldas_mask_difference'); plt.show()
print('Common mask of all 12 GLDAS variables:', np.logical_and.reduce(list(sets.values())).sum(), 'cells')

## 3. Value distributions

A sample of up to 200 k values per variable (0.5–99.5 percentile range). Shows scale, skew, share of exact zeros (snow, precipitation) and potential outliers.

In [ ]:
rng = np.random.default_rng(0)
fig, axs = plt.subplots(5,4, figsize=(18,15))
for ax,k in zip(axs.ravel(), KEYS):
    c = np.asarray(load_cube(k)[0]); v = c[np.isfinite(c)]
    v = rng.choice(v, size=min(200_000, v.size), replace=False)
    lo,hi = np.percentile(v,[0.5,99.5]); ax.hist(v[(v>=lo)&(v<=hi)], bins=70, color=PKG_COLOR[pkg(k)], alpha=.9)
    ax.set_title(f'{k}  [{U[k]}]', fontsize=8.5); ax.set_yticks([])
    ax.text(.98,.9,f'zeros: {(v==0).mean()*100:.0f}%', transform=ax.transAxes, ha='right', fontsize=7.5, color=PAL['grey'])
title(fig, 'Value distributions (0.5–99.5 percentiles)'); plt.tight_layout(); save_fig(fig, 'distributions'); plt.show()

## 4. Global time series and trends

Monthly **area-weighted mean** (weight `cos φ`). Two versions:
* **Native domain**: GLDAS = land only; IMERG & MERRA-2 = whole globe (not comparable across products);
* **Land-only (common mask)**: every variable averaged over the same GLDAS land cells, which makes products directly comparable.

In [ ]:
def wmean(cube, mask=None):
    w = W if mask is None else W*mask
    out = np.full(T, np.nan)
    for t in range(T):
        x = np.asarray(cube[t]); m = np.isfinite(x) & (w>0)
        if m.any(): out[t] = (x[m]*w[m]).sum()/w[m].sum()
    return out

def deseason(series):
    a = np.asarray(series).reshape(NY,12); return (a - np.nanmean(a,axis=0)).ravel()

GM  = {k: cached(f'gm_{k}',  lambda k=k: wmean(load_cube(k)[0]))       for k in KEYS}   # native domain
GML = {k: cached(f'gml_{k}', lambda k=k: wmean(load_cube(k)[0], LAND)) for k in KEYS}   # land-only
x = MONTHS.to_timestamp()
fig, axs = plt.subplots(5,4, figsize=(18,15), sharex=True)
for ax,k in zip(axs.ravel(), KEYS):
    ax.plot(x, GM[k], lw=.7, color=PKG_COLOR[pkg(k)], alpha=.55)
    ax.plot(x, pd.Series(GM[k], index=x).rolling(12, min_periods=9).mean(), lw=1.8, color=PAL['navy'])
    ax.set_title(f'{k}  [{U[k]}]', fontsize=8.5)
title(fig, 'Global mean, native domain (thin: monthly, bold: 12-month rolling mean)'); plt.tight_layout(); save_fig(fig, 'global_series'); plt.show()

### Trend functions
Anomalies → annual means → OLS. Significance is based on **26 annual values**, not 312 months, because monthly anomalies are autocorrelated and would give overly optimistic p-values. `sigma/decade` expresses the slope in units of the standard deviation of the annual anomalies, which makes variables with different units comparable.

In [ ]:
def trend_summary(series):
    an = deseason(series); ann = np.nanmean(an.reshape(NY,12), axis=1)
    xx = (YEARS-2000)/10.0; ok = np.isfinite(ann); n = int(ok.sum())
    if n < 8:
        return dict(slope=np.nan, ci_lo=np.nan, ci_hi=np.nan, p_ols=np.nan, theil_sen=np.nan, mk_tau=np.nan, mk_p=np.nan, sigma=np.nan, n_years=n), ann
    lr = stats.linregress(xx[ok], ann[ok]); tc = stats.t.ppf(.975, n-2)
    ts = stats.theilslopes(ann[ok], xx[ok]); tau,pmk = stats.kendalltau(xx[ok], ann[ok])
    sd = np.nanstd(ann)
    return dict(slope=lr.slope, ci_lo=lr.slope-tc*lr.stderr, ci_hi=lr.slope+tc*lr.stderr, p_ols=lr.pvalue,
                theil_sen=ts[0], mk_tau=tau, mk_p=pmk, sigma=lr.slope/sd if sd>0 else np.nan, n_years=n), ann

def bh(p):
    p = np.asarray(p, float); out = np.full(len(p), np.nan); ok = np.isfinite(p)
    q = p[ok]; n = len(q); o = np.argsort(q); r = q[o]*n/(np.arange(n)+1)
    r = np.minimum.accumulate(r[::-1])[::-1]; z = np.empty(n); z[o] = np.minimum(r,1); out[ok] = z; return out

def trend_table(series_dict):
    rows = {}
    for k,s in series_dict.items():
        d,_ = trend_summary(s); d['mean'] = np.nanmean(s); d['units'] = U[k]; rows[k] = d
    df = pd.DataFrame(rows).T
    for c in df.columns:
        if c != 'units': df[c] = pd.to_numeric(df[c])
    df['q_fdr'] = bh(df['p_ols'].values)
    df['pct/decade'] = np.where(df['mean'].abs()>1e-9, df['slope']/df['mean'].abs()*100, np.nan)
    df['significant (FDR 5%)'] = np.where(df['q_fdr']<0.05,'yes','no')
    return df[['units','mean','slope','ci_lo','ci_hi','sigma','pct/decade','p_ols','q_fdr','mk_p','theil_sen','significant (FDR 5%)']]

display(Markdown('**Native domain** (GLDAS = land, IMERG/MERRA-2 = globe)'))
gt = trend_table(GM); display(nice(gt, gradient=['sigma'], cmap='RdBu_r'))
display(Markdown('**Land-only, common mask** (all variables on the same GLDAS land cells)'))
glt = trend_table(GML); display(nice(glt, gradient=['sigma'], cmap='RdBu_r'))

In [ ]:
fig, axs = plt.subplots(5,4, figsize=(18,15), sharex=True)
xx = (YEARS-2000)/10
for ax,k in zip(axs.ravel(), KEYS):
    d, ann = trend_summary(GML[k]); ok = np.isfinite(ann)
    ax.plot(YEARS, ann, 'o-', ms=3, lw=.9, color=PKG_COLOR[pkg(k)])
    b = np.polyfit(xx[ok], ann[ok], 1); sig = glt.loc[k,'significant (FDR 5%)']=='yes'
    ax.plot(YEARS, np.polyval(b,xx), '--', lw=1.6, color=PAL['red'] if sig else PAL['grey'])
    ax.set_title(f'{k}   {d["slope"]:+.3g} {U[k]}/dec   p={d["p_ols"]:.3f}{"  ★" if sig else ""}', fontsize=8.5)
title(fig, 'Land-only global mean: annual anomalies and linear trend   (★ = significant after FDR)'); plt.tight_layout(); save_fig(fig, 'global_land_trends'); plt.show()

## 5. Climate zones: trends by latitude band

The globe is split into five latitude bands. Each variable is averaged over the **land cells** of each band (common GLDAS mask, so the bands are comparable across products). Bands without land data (e.g. Antarctica, which GLDAS does not cover) stay empty.

The heat-map shows the trend in **σ per decade**; ★ marks tests that stay significant after FDR correction over all variable × band combinations.

In [ ]:
BANDS = {'Arctic (>66.5°N)':               LATM>66.5,
         'N. mid-latitudes (23.5–66.5°N)': (LATM>23.5)&(LATM<=66.5),
         'Tropics (23.5°S–23.5°N)':        np.abs(LATM)<=23.5,
         'S. mid-latitudes (23.5–66.5°S)': (LATM<-23.5)&(LATM>=-66.5),
         'Antarctic (<66.5°S)':            LATM<-66.5}
rows = []
for k in KEYS:
    for bi,(bn,bm) in enumerate(BANDS.items()):
        s = cached(f'band_{k}_{bi}', lambda k=k,bm=bm: wmean(load_cube(k)[0], LAND & bm))
        if np.isfinite(s).sum() < 120: rows.append(dict(variable=k, band=bn, slope=np.nan, sigma=np.nan, p=np.nan, cells=int((LAND&bm).sum()))); continue
        d,_ = trend_summary(s); rows.append(dict(variable=k, band=bn, slope=d['slope'], sigma=d['sigma'], p=d['p_ols'], cells=int((LAND&bm).sum())))
BT = pd.DataFrame(rows); BT['q'] = bh(BT['p'].values)
SIG = BT.pivot(index='variable', columns='band', values='sigma').reindex(index=KEYS, columns=list(BANDS))
SLP = BT.pivot(index='variable', columns='band', values='slope').reindex(index=KEYS, columns=list(BANDS))
QQ  = BT.pivot(index='variable', columns='band', values='q').reindex(index=KEYS, columns=list(BANDS))
fig, ax = plt.subplots(figsize=(11,8.5))
im = ax.imshow(SIG.values.astype(float), cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto'); ax.grid(False)
ax.set_xticks(range(len(BANDS))); ax.set_xticklabels([b.replace(' (','\n(') for b in BANDS], fontsize=8)
ax.set_yticks(range(len(KEYS))); ax.set_yticklabels(KEYS, fontsize=8.5)
for i in range(len(KEYS)):
    for j in range(len(BANDS)):
        v = SIG.values[i,j]
        if np.isfinite(v): ax.text(j, i, f'{v:+.1f}'+('★' if QQ.values[i,j]<0.05 else ''), ha='center', va='center', fontsize=7.5, color='black')
cb = plt.colorbar(im, ax=ax, shrink=.8, pad=.02); cb.set_label('trend, σ per decade'); cb.outline.set_visible(False)
ax.set_title('Trend by latitude band, land cells (★ significant after FDR)', fontsize=12, color=PAL['navy'], fontweight='bold'); save_fig(fig, 'latitude_band_trends'); plt.show()
display(Markdown('**Slopes in native units per decade**'))
display(nice(SLP.assign(units=[U[k] for k in SLP.index])))
print(f'Significant (FDR 5%): {(QQ.values<0.05).sum()} of {np.isfinite(QQ.values).sum()} variable × band tests')

## 6. Zonal structure: latitude × time

Zonal-mean (averaged over longitude) annual anomalies. A Hovmöller diagram reveals when and at which latitudes the signal changes, which a single global mean hides.

In [ ]:
def zonal_anom(k):
    a = np.asarray(load_cube(k)[0], dtype=np.float32).reshape(NY,12,180,360)
    an = a - np.nanmean(a,axis=0); ann = np.nanmean(an,axis=1)
    return np.nanmean(ann, axis=2)      # (26 years, 180 lats)

HOV = ['merra2_t2m','imerg_precip','merra2_tqv','gldas_soilmoist_0_10']
ZH = {k: cached(f'zh_{k}', lambda k=k: zonal_anom(k)) for k in HOV}
fig, axs = plt.subplots(2,2, figsize=(16,8))
for ax,k in zip(axs.ravel(), HOV):
    z = ZH[k].T; v = np.nanpercentile(np.abs(z), 98)
    im = ax.imshow(z, origin='lower', extent=[2000,2026,-90,90], aspect='auto', cmap='RdBu_r', vmin=-v, vmax=v, interpolation='nearest')
    ax.set_title(f'{k}: zonal-mean annual anomaly [{U[k]}]'); ax.grid(False); ax.set_ylabel('latitude')
    cb = plt.colorbar(im, ax=ax, shrink=.9, pad=.02); cb.outline.set_visible(False)
title(fig, 'Hovmöller diagrams (latitude × year)'); plt.tight_layout(); save_fig(fig, 'hovmoeller'); plt.show()

## 7. Global trend maps (every grid cell)

For **each cell** separately: anomalies → annual means → OLS over 26 years. Grey overlay = **not significant** (p ≥ 0.05).
Caution: this is 64 800 independent tests, so ~5 % of cells are expected to look "significant" by chance (field-significance problem). The maps show a **spatial pattern**, not proof for any single cell.

In [ ]:
XD = ((YEARS-2000)/10.0)[:,None,None]
def trend_maps(cube):
    a = np.asarray(cube, dtype=np.float32).reshape(NY,12,180,360)
    an = a - np.nanmean(a,axis=0); ann = np.nanmean(an,axis=1)
    m = np.isfinite(ann); n = m.sum(0).astype(float); y0 = np.where(m,ann,0.0)
    with np.errstate(all='ignore'):
        xm = (XD*m).sum(0)/n; ym = y0.sum(0)/n
        sxx = (((XD-xm)**2)*m).sum(0); sxy = ((XD-xm)*(y0-ym)*m).sum(0)
        sl = sxy/sxx; sse = ((((y0-ym)-sl*(XD-xm))**2)*m).sum(0)
        se = np.sqrt(sse/(n-2)/sxx); tt = sl/se; p = 2*stats.t.sf(np.abs(tt), n-2)
    bad = n < 20; sl[bad] = np.nan; p[bad] = np.nan
    return sl.astype(np.float32), p.astype(np.float32)

SLOPE, PVAL = {}, {}
for k in KEYS: SLOPE[k], PVAL[k] = cached(f'trend_{k}', lambda k=k: trend_maps(load_cube(k)[0]))
np.savez_compressed(ng.DATA/'analysis'/'trend_maps.npz', **{f'{k}_slope':SLOPE[k] for k in KEYS}, **{f'{k}_p':PVAL[k] for k in KEYS})
rows = {}
for k in KEYS:
    s,p = SLOPE[k], PVAL[k]; m = np.isfinite(s)
    rows[k] = {'units / decade':U[k], 'cells':int(m.sum()), 'area-weighted mean slope':(s[m]*W[m]).sum()/W[m].sum(),
               '% cells positive':(s[m]>0).mean()*100, '% cells significant (p<0.05)':(p[m]<0.05).mean()*100}
display(nice(pd.DataFrame(rows).T.astype({'cells':int}), gradient=['% cells significant (p<0.05)']))

In [ ]:
show = ['gldas_tair','merra2_t2m','gldas_precip','imerg_precip','gldas_soilmoist_0_10','gldas_evap','merra2_tqv','gldas_swe']
fig, axs = plt.subplots(4,2, figsize=(17,17))
for ax,k in zip(axs.ravel(), show):
    s = SLOPE[k]; v = np.nanpercentile(np.abs(s), 98) or 1
    show_map(ax, s, f'{k}: trend [{U[k]} / decade]', cmap='RdBu_r', vmin=-v, vmax=v)
    ns = np.where(np.isfinite(PVAL[k]) & (PVAL[k] >= 0.05), 1.0, np.nan)
    ax.imshow(ns, origin='lower', extent=[-180,180,-90,90], cmap=ListedColormap(['#c9ced6']), alpha=.6, aspect='auto', interpolation='nearest')
title(fig, 'Per-cell linear trends, 2000–2025 (grey = not significant)'); plt.tight_layout(); save_fig(fig, 'trend_maps'); plt.show()

## 8. Cross-validation between products

Some quantities are produced independently by two sources. If their **changes** agree, the grid can be trusted. We compare **anomalies** (absolute levels naturally differ: different models, elevations, averaging) on cells where both exist.

Pairs: air temperature (GLDAS vs MERRA-2), surface temperature, precipitation (GLDAS vs IMERG), wind speed (GLDAS vs √(u²+v²) from MERRA-2).

In [ ]:
def anom_cube(c):
    a = np.asarray(c, dtype=np.float32).reshape(NY,12,180,360); return (a - np.nanmean(a,axis=0)).reshape(T,180,360)

def corr_map(a, b):
    m = np.isfinite(a) & np.isfinite(b); n = m.sum(0).astype(float)
    a0 = np.where(m,a,0.0); b0 = np.where(m,b,0.0)
    with np.errstate(all='ignore'):
        ma = a0.sum(0)/n; mb = b0.sum(0)/n
        da = (a0-ma)*m; db = (b0-mb)*m
        r = (da*db).sum(0)/np.sqrt((da**2).sum(0)*(db**2).sum(0))
    r[n<100] = np.nan; return r

def wind_cube():
    u = np.asarray(load_cube('merra2_u10m')[0]); v = np.asarray(load_cube('merra2_v10m')[0]); return np.hypot(u,v)

PAIRS = [('gldas_tair','merra2_t2m','Air temperature, °C'),
         ('gldas_skin_temp','merra2_ts','Surface temperature, °C'),
         ('gldas_precip','imerg_precip','Precipitation, mm/month'),
         ('gldas_wind','WIND_M2','10 m wind speed, m/s')]

def corr_pair(ka, kb):
    A = anom_cube(load_cube(ka)[0]); B = anom_cube(wind_cube() if kb=='WIND_M2' else load_cube(kb)[0])
    return corr_map(A,B)

RESULT = {}
fig, axs = plt.subplots(2,2, figsize=(17,8.5))
for ax,(ka,kb,name) in zip(axs.ravel(), PAIRS):
    r = cached(f'corr_{ka}_{kb}', lambda ka=ka, kb=kb: corr_pair(ka,kb)); RESULT[(ka,kb)] = r
    show_map(ax, r, f'{name}: correlation of anomalies, {ka} ~ {kb}', cmap='RdYlGn', vmin=-1, vmax=1, label='Pearson r')
title(fig, 'Month-to-month agreement between independent products'); plt.tight_layout(); save_fig(fig, 'cross_correlation'); plt.show()
rows = {f'{a} ~ {b}': {'cells':int(np.isfinite(r).sum()), 'median r':np.nanmedian(r), 'area-weighted mean r':np.nansum(r*W)/W[np.isfinite(r)].sum(),
         '% cells r>0.5':np.nanmean(r[np.isfinite(r)]>0.5)*100, '% cells r<0':np.nanmean(r[np.isfinite(r)]<0)*100} for (a,b),r in RESULT.items()}
display(nice(pd.DataFrame(rows).T.astype({'cells':int}), gradient=['median r'], cmap='Greens'))

In [ ]:
# Do the products agree on TRENDS? Per-cell slopes, product A vs product B
SLOPE['WIND_M2'], PVAL['WIND_M2'] = cached('trend_WIND_M2', lambda: trend_maps(wind_cube()))
fig, axs = plt.subplots(1,4, figsize=(19,4.6)); out = {}
for ax,(ka,kb,name) in zip(axs, PAIRS):
    a, b = SLOPE[ka], SLOPE[kb]; m = np.isfinite(a)&np.isfinite(b)
    ax.scatter(a[m][::7], b[m][::7], s=3, alpha=.25, color=PAL['navy']); lim = np.nanpercentile(np.abs(np.r_[a[m],b[m]]), 99)
    ax.plot([-lim,lim],[-lim,lim], '--', lw=1.4, color=PAL['red']); ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim)
    r = np.corrcoef(a[m],b[m])[0,1]; same = (np.sign(a[m])==np.sign(b[m])).mean()*100
    ax.set_title(f'{name}\nr = {r:.2f}, same sign: {same:.0f}%', fontsize=9); ax.set_xlabel(ka); ax.set_ylabel(kb)
    out[f'{ka} vs {kb}'] = {'r of slopes':r, '% same sign':same, 'mean slope A':np.nansum(a[m]*W[m])/W[m].sum(), 'mean slope B':np.nansum(b[m]*W[m])/W[m].sum()}
title(fig, 'Per-cell trend agreement (red = 1:1)'); plt.tight_layout(); save_fig(fig, 'cross_trend_agreement'); plt.show(); display(nice(pd.DataFrame(out).T))

In [ ]:
# Zonal-mean trend profiles: do the products agree on WHERE the changes happen?
fig, axs = plt.subplots(1,4, figsize=(19,4.6), sharey=True)
for ax,(ka,kb,name) in zip(axs, PAIRS):
    for kk,col in [(ka,PAL['teal']),(kb,PAL['plum'])]:
        ax.plot(np.nanmean(SLOPE[kk],axis=1), LAT, color=col, lw=1.8, label=kk)
    ax.axvline(0, color=PAL['grey'], lw=.8); ax.set_title(name, fontsize=9); ax.legend(fontsize=7.5); ax.set_xlabel('trend / decade')
axs[0].set_ylabel('latitude')
title(fig, 'Zonal-mean trend profiles'); plt.tight_layout(); save_fig(fig, 'zonal_trend_profiles'); plt.show()

# Land-only global annual anomalies for the same pairs
fig, axs = plt.subplots(1,3, figsize=(18,4))
GML['WIND_M2'] = cached('gml_WIND_M2', lambda: wmean(wind_cube(), LAND)); U['WIND_M2'] = 'm s-1'
for ax,(ka,kb,name) in zip(axs, PAIRS[:3]):
    _,aa = trend_summary(GML[ka]); _,bb = trend_summary(GML[kb])
    ax.plot(YEARS, aa, 'o-', ms=3.5, color=PAL['teal'], label=ka); ax.plot(YEARS, bb, 's-', ms=3.5, color=PAL['plum'], label=kb); ax.legend(fontsize=8)
    ok = np.isfinite(aa)&np.isfinite(bb); ax.set_title(f'{name}, global land: r = {np.corrcoef(aa[ok],bb[ok])[0,1]:.2f}', fontsize=9)
title(fig, 'Global land annual anomalies: two products, one signal?'); plt.tight_layout(); save_fig(fig, 'global_land_pairs'); plt.show()

## 9. Data-quality audit

Plausibility versus physical limits, impossible values (negatives where they should not exist), fully missing months, and the distribution of `valid_frac` (the share of a cell covered by valid source data).

In [ ]:
LIM = {'gldas_tair':(-90,60),'merra2_t2m':(-90,60),'gldas_skin_temp':(-90,90),'merra2_ts':(-90,90),
       'merra2_ps':(30000,110000),'merra2_qv2m':(0,0.04),'merra2_tqv':(0,100),
       'gldas_wind':(0,60),'merra2_u10m':(-60,60),'merra2_v10m':(-60,60),
       'gldas_precip':(0,3000),'imerg_precip':(0,3000),'gldas_swdown':(0,500),'gldas_potevap':(-50,1000),
       'gldas_swe':(0,20000),'gldas_snowdepth':(0,50),'gldas_soilmoist_0_10':(0,100),'gldas_soilmoist_10_40':(0,300),
       'gldas_rootmoist':(0,500),'gldas_evap':(-50,400)}
rows = {}
for k in KEYS:
    c = np.asarray(load_cube(k)[0]); v = c[np.isfinite(c)]; lo,hi = LIM[k]
    rows[k] = {'plausible range':f'[{lo}, {hi}]','below':int((v<lo).sum()),'above':int((v>hi).sum()),
               'negative values':int((v<0).sum()),'% negative':(v<0).mean()*100,'min':float(v.min()),'max':float(v.max())}
display(nice(pd.DataFrame(rows).T.astype({'below':int,'above':int,'negative values':int})))

In [ ]:
# Fully missing months (all cells NaN)
any_missing = False
for k in KEYS:
    c = load_cube(k)[0]; miss = [str(MONTHS[t]) for t in range(T) if not np.isfinite(np.asarray(c[t])).any()]
    if miss: any_missing = True; print(f'{k}: no data in months {miss}')
if not any_missing: print('✔ No fully missing months')
fig, axs = plt.subplots(1,3, figsize=(17,3.8))
for ax,(name,ks) in zip(axs, [('GLDAS',[k for k in KEYS if k.startswith('gldas')]),('IMERG',['imerg_precip']),('MERRA-2',[k for k in KEYS if k.startswith('merra2')])]):
    for k in ks:
        f = FRAC[k][np.isfinite(FRAC[k])]; ax.hist(f, bins=50, range=(0.5,1), alpha=.4, color=PKG_COLOR[pkg(k)])
    ax.set_title(f'{name}: mean valid_frac per cell'); ax.set_yscale('log'); ax.set_xlabel('valid_frac')
title(fig, 'Share of each cell covered by valid source data'); plt.tight_layout(); save_fig(fig, 'valid_frac'); plt.show()
for k in ['gldas_evap','imerg_precip','merra2_t2m']:
    f = FRAC[k][np.isfinite(FRAC[k])]; print(f'{k}: valid_frac < 1 in {(f<0.999).mean()*100:.1f}% of cells (coastlines, poles, source-grid edges)')

## 10. Summary (assembled automatically from the results above)

In [ ]:
print('=== INTEGRITY ===')
print(f'Variables: {len(KEYS)} | total CSV size: {sum(s["size_mb"] for s in STATS.values())/1000:.1f} GB | rows: {sum(s["rows"] for s in STATS.values()):,}')
print(f'Duplicates: {sum(s["dup"] for s in STATS.values())} | rows with bad dates: {sum(s["bad_dates"] for s in STATS.values())}')
print('Problems:', problems if problems else 'none')
print('\n=== GLOBAL LAND TRENDS (land-only mean, per decade) ===')
for k in KEYS:
    r = glt.loc[k]; flag = '★' if r['significant (FDR 5%)']=='yes' else ' '
    print(f'{flag} {k:24s} {r["slope"]:+10.4g} {U[k]:12s} (95% CI {r["ci_lo"]:+.3g} … {r["ci_hi"]:+.3g}, q={r["q_fdr"]:.3f})')
print('\n=== LATITUDE BANDS ===')
print(f'Significant variable × band tests (FDR 5%): {(QQ.values<0.05).sum()} of {np.isfinite(QQ.values).sum()}')
print('\n=== CROSS-VALIDATION (median r of anomalies) ===')
for (a,b),r in RESULT.items(): print(f'{a} ~ {b}: {np.nanmedian(r):.2f}')